# Pharmacy Sales Forecasting by Drug Group

**Question:** how much of each drug group will a pharmacy sell each week next year, so it can plan stock?

**Data:** `salesweekly.csv` from [Pharma Sales Data](https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data) on Kaggle (free account). Save it in `data/` (see `data/README.md`).

## 1. Load and explore

In [ ]:
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose   # splits a series into trend + season + rest
from statsmodels.tsa.statespace.sarimax import SARIMAX    # the ARIMA model
from prophet import Prophet                               # Meta's forecasting library
logging.getLogger('cmdstanpy').disabled = True   # quieter Prophet

# parse_dates turns the 'datum' (date) column into real dates, which then become the row labels
df = pd.read_csv('data/salesweekly.csv', parse_dates=['datum']).set_index('datum').sort_index()
print(df.columns.tolist())
print(df.tail(3))   # is the last week far lower than the rest? Then it is incomplete:
df = df.iloc[:-1]   # drop the last row

series = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']   # the 8 ATC groups
# One small line chart per group, in a grid of 4 rows and 2 columns
df[series].plot(subplots=True, layout=(4, 2), figsize=(14, 10), title='Weekly sales by ATC group')
plt.tight_layout()
plt.show()

# period=52: in weekly data the pattern repeats every 52 weeks (one year)
seasonal_decompose(df['R06'], model='additive', period=52).plot()   # trend + yearly pattern
plt.show()

## 2. Seasonality

In [ ]:
import seaborn as sns

# The same decomposition for every group; suptitle writes the group's code above each chart
for col in series:
    seasonal_decompose(df[col], model='additive', period=52).plot()
    plt.suptitle(col, y=1.02)
    plt.show()

# Average weekly sales in each calendar month, scaled so each group's average = 100
monthly = df[series].groupby(df.index.month).mean()   # rows = months 1-12, columns = groups
seasonal_index = 100 * monthly / monthly.mean()      # 120 = 20% above that group's average month

plt.figure(figsize=(10, 4))
# .T flips the table so groups become rows; center=100 makes 100 white, higher red and lower blue
sns.heatmap(seasonal_index.T, annot=True, fmt='.0f', cmap='RdBu_r', center=100)
plt.title('Seasonal index by month (100 = an average month)')
plt.xlabel('Month')
plt.tight_layout()
plt.savefig('images/seasonal_index.png', dpi=150)

**What I found:** _which groups are seasonal, and in which months do they peak? Why, clinically?_

## 3. Hold-out split and error metrics

In [ ]:
H = 52                                    # forecast one full year
train, test = df.iloc[:-H], df.iloc[-H:]  # all weeks except the last 52, and the last 52

def mape(y, yhat):
    """Mean absolute percentage error. Weeks with zero sales are skipped (they would divide by zero)."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    mask = y != 0
    return 100 * np.mean(np.abs((y[mask] - yhat[mask]) / y[mask]))

def wape(y, yhat):
    """Weighted absolute percentage error: total absolute error divided by total sales."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    return 100 * np.abs(y - yhat).sum() / np.abs(y).sum()

def fourier(t, period=52.18, K=3):
    # sine/cosine waves that let ARIMA learn a yearly pattern in weekly data.
    # t = week numbers, 52.18 = average weeks in a year, K = 3 pairs of waves (smooth but flexible)
    return np.column_stack([f(2 * np.pi * k * t / period) for k in range(1, K + 1) for f in (np.sin, np.cos)])

# Week numbers: 0, 1, 2 ... for the training weeks, then the numbers that follow for the test weeks
t_train, t_test = np.arange(len(train)), np.arange(len(train), len(df))

## 4. Three models for every drug group

In [ ]:
results, forecasts = [], {}   # error scores for the table, and forecasts for the charts
for col in series:            # one drug group at a time
    y_tr, y_te = train[col].values, test[col].values
    preds = {'Seasonal naive': y_tr[-H:]}   # same week last year

    # ARIMA(1,1,1), with the Fourier waves as extra inputs (exog) for the yearly pattern
    arima = SARIMAX(y_tr, exog=fourier(t_train), order=(1, 1, 1)).fit(disp=False)
    preds['ARIMA'] = arima.forecast(steps=H, exog=fourier(t_test))

    # Prophet needs a table with a 'ds' (date) column and a 'y' (value) column
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(pd.DataFrame({'ds': train.index, 'y': y_tr}))
    preds['Prophet'] = m.predict(pd.DataFrame({'ds': test.index}))['yhat'].values   # yhat = forecast

    for model, yhat in preds.items():
        yhat = np.clip(yhat, 0, None)       # sales can't be negative
        preds[model] = yhat
        results.append({'series': col, 'model': model,
                        'WAPE': wape(y_te, yhat), 'MAPE': mape(y_te, yhat)})
    forecasts[col] = preds

res = pd.DataFrame(results)
# Reshape: one row per drug group, one column per model
res.pivot(index='series', columns='model', values='WAPE').round(1)

## 5. Charts

In [ ]:
col = 'N02BE'   # change this to chart another group
plt.figure(figsize=(12, 4))
plt.plot(train.index[-104:], train[col].iloc[-104:], label='History')   # the last 2 years of training data
plt.plot(test.index, test[col], label='Actual', color='black')
for model, yhat in forecasts[col].items():   # one dashed line per model
    plt.plot(test.index, yhat, label=model, linestyle='--')
plt.legend()
plt.title(f'{col}: 52-week hold-out forecast')
plt.tight_layout()
plt.savefig(f'images/forecast_{col}.png', dpi=150)

In [ ]:
# Bars grouped by drug group, one bar per model
wape_table = res.pivot(index='series', columns='model', values='WAPE')
wape_table.plot(kind='bar', figsize=(10, 4))
plt.ylabel('WAPE (%), 52-week hold-out')
plt.title('Forecast error by drug group and model (lower is better)')
plt.tight_layout()
plt.savefig('images/wape_by_model.png', dpi=150)

# True for each group where the better of ARIMA and Prophet has a lower error than seasonal naive
beats = wape_table[['ARIMA', 'Prophet']].min(axis=1) < wape_table['Seasonal naive']
print(f'A model beat seasonal naive in {beats.sum()} of {len(beats)} groups')   # sum() counts the Trues

**What I found:** _in how many groups did a model beat seasonal naive, and what was the best WAPE?_

## 6. 12-week stock plan

The best model for each group is refitted on all the data, including the hold-out year.

In [ ]:
best_model = wape_table[['ARIMA', 'Prophet']].idxmin(axis=1)   # the better of the two per group
future_dates = df.index[-1] + pd.to_timedelta(7 * np.arange(1, 13), unit='D')   # next 12 weeks
t_all, t_future = np.arange(len(df)), np.arange(len(df), len(df) + 12)   # week numbers: all data, next 12

plan = {}
for col in series:
    if best_model[col] == 'ARIMA':
        # Refit ARIMA on all the data, then forecast 12 weeks ahead
        fit = SARIMAX(df[col].values, exog=fourier(t_all), order=(1, 1, 1)).fit(disp=False)
        plan[col] = fit.forecast(steps=12, exog=fourier(t_future))
    else:
        # Refit Prophet on all the data, then forecast the 12 future dates
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        m.fit(pd.DataFrame({'ds': df.index, 'y': df[col].values}))
        plan[col] = m.predict(pd.DataFrame({'ds': future_dates}))['yhat'].values

# One row per future week, one column per drug group; negative forecasts become 0
stock_plan = pd.DataFrame(plan, index=future_dates).clip(lower=0).round(1)
stock_plan.to_csv('stock_plan_next_12_weeks.csv')
stock_plan